W poniższym notebooku przedstawiono kod, który rozpakowuje pliki .h5 z danymi muzycznymi i zapisuje wybrane atrybuty (`columns_to_keep`) do plików CSV.

In [1]:
import os
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
main_directory = '../DATA/raw/extracted_files'

uwaga: zmiany 29.12.2025

In [ ]:
import os
import h5py
import numpy as np
import glob
import pandas as pd
from collections import OrderedDict
from tqdm import tqdm

# 'bars', 'beats', 'tatums', 'sections' - to głównie znaczniki czasowe, które nie przydają się do predykcji popularności
DROP_KEYWORDS = ["bars", "beats", "tatums", "sections", "idx"]

def should_keep(name: str) -> bool:
    # zwraca False jeśli nazwa zawiera zakazane słowo
    return not any(k in name.lower() for k in DROP_KEYWORDS)

def extract_all_attributes(h5_file):
    row = OrderedDict()

    def collect(name, obj):
        if not should_keep(name):
            return

        if isinstance(obj, h5py.Dataset):
            value = obj[()]

            if isinstance(value, bytes):
                value = value.decode('utf-8', errors='ignore')

            elif isinstance(value, np.ndarray):
                if hasattr(value, "dtype") and value.dtype.names and value.shape == (1,):
                    record = value[0]
                    for key in record.dtype.names:
                        full_key = f"{name}/{key}"
                        if not should_keep(full_key):
                            continue
                        val = record[key]
                        if isinstance(val, bytes):
                            val = val.decode('utf-8', errors='ignore')
                        row[full_key] = val
                    return
                else:
                    value = value.tolist()

            row[name] = value

    h5_file.visititems(collect)
    return row

def process_folder(folder_path):
    rows = []
    
    pattern = os.path.join(folder_path, "**", "*.h5")
    files = glob.glob(pattern, recursive=True)

    print(f"Znaleziono {len(files)} plików .h5 (we wszystkich podfolderach)\n")

    for file_path in tqdm(files, desc="Przetwarzanie plików"):
        try:
            with h5py.File(file_path, "r") as f:
                row = extract_all_attributes(f)
                rows.append(row)
        except Exception as e:
            print(f"{file_path} - {e}")

    return pd.DataFrame(rows)

Kolumny z przetworzonego subsetu:

In [ ]:
columns_to_keep = [
    'metadata/songs/song_id',
    'metadata/songs/title',
    'metadata/songs/artist_name',
    'metadata/songs/artist_id',
    'metadata/songs/release', # nazwa albumu
    'musicbrainz/songs/year',          
    
    'metadata/songs/song_hotttnesss',  
    'metadata/songs/artist_hotttnesss',
    'metadata/songs/artist_familiarity',
    'metadata/songs/artist_location',

    'analysis/songs/duration',
    'analysis/songs/loudness',
    'analysis/songs/tempo',
    'analysis/songs/key',
    'analysis/songs/key_confidence',
    'analysis/songs/mode',             # dur/moll
    'analysis/songs/mode_confidence',
    'analysis/songs/time_signature',
    'analysis/songs/time_signature_confidence',
    'analysis/songs/end_of_fade_in',
    'analysis/songs/start_of_fade_out',

    'metadata/artist_terms',           # gatunki/tagi (Echo Nest)
    'metadata/artist_terms_freq',
    'metadata/artist_terms_weight',
    'musicbrainz/artist_mbtags',       # tagi (MusicBrainz)
    'metadata/similar_artists',        

    'analysis/segments_loudness_max',  # do dynamiki utworu
    'analysis/segments_pitches',       # do wariancji melodii
    'analysis/segments_timbre'         # do barwy dźwięku (jasny/ciemny)
]

In [8]:
directory_A = rf'{main_directory}\A'
df_A = process_folder(directory_A)

Znaleziono 14896 plików .h5 (we wszystkich podfolderach)



Przetwarzanie plików: 100%|██████████| 14896/14896 [26:40<00:00,  9.31it/s]   


In [10]:
df_A.drop(columns=[col for col in df_A.columns if col not in columns_to_keep], inplace=True)

In [11]:
os.makedirs('extracted_DataFrames', exist_ok=True)
df_A.to_csv('extracted_DataFrames/songs_A.csv', index=False)

In [ ]:
#letters = ['B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K']
letters = ['C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K']
for letter in letters:
    print(f'Processing directory: {letter}')
    directory = rf'{main_directory}\{letter}'
    df = process_folder(directory)
    df.drop(columns=[col for col in df.columns if col not in columns_to_keep], inplace=True)
    df.to_csv(f'../DATA/semi-processed/extracted_DataFrames/songs_{letter}.csv', index=False)

Processing directory: C
Znaleziono 14637 plików .h5 (we wszystkich podfolderach)



Przetwarzanie plików: 100%|██████████| 14637/14637 [17:22<00:00, 14.04it/s]   


Processing directory: D
Znaleziono 14901 plików .h5 (we wszystkich podfolderach)



Przetwarzanie plików: 100%|██████████| 14901/14901 [49:03<00:00,  5.06it/s]   


Processing directory: E
Znaleziono 14637 plików .h5 (we wszystkich podfolderach)



...

Przetwarzanie plików: 100%|██████████| 14951/14951 [1:24:40<00:00,  2.94it/s]     


Processing directory: J
Znaleziono 14859 plików .h5 (we wszystkich podfolderach)



Przetwarzanie plików: 100%|██████████| 14859/14859 [1:12:15<00:00,  3.43it/s]    


Processing directory: K
Znaleziono 14962 plików .h5 (we wszystkich podfolderach)



Przetwarzanie plików: 100%|██████████| 14962/14962 [2:07:48<00:00,  1.95it/s]    


In [7]:
print(f'Processing directory: K')
directory = rf'{main_directory}\K'
df = process_folder(directory)
df.drop(columns=[col for col in df.columns if col not in columns_to_keep], inplace=True)
df.to_csv(f'../DATA/semi-processed/extracted_DataFrames/songs_K.csv', index=False)

Processing directory: K
Znaleziono 14962 plików .h5 (we wszystkich podfolderach)



Przetwarzanie plików: 100%|██████████| 14962/14962 [17:21<00:00, 14.37it/s]   
